# Dragonfly g/r cirrus modeling

This notebook builds a shared luminance image from source-subtracted Dragonfly g- and r-band images, applies the RHT morphology filter, and predicts a cirrus model in each band.

In [1]:
from pathlib import Path

from dfcirrus.modeling import ModelingConfig, MultiBandModeler
from dfcirrus.modeling.diagnostics import (
    plot_interband_fits,
    plot_model_images,
    plot_morphology,
    plot_planck_fits,
)

## Input files and calibration

Set the image paths and update the calibration values if they differ from the defaults. Images are expected to contain calibrated counts with celestial WCS headers.

In [ ]:
G_IMAGE = Path("data/field-g.fits").resolve()
R_IMAGE = Path("data/field-r.fits").resolve()
PLANCK_MAP = Path("/Users/qliu/data/HFI_CompMap_ThermalDustModel_2048_R1.20.fits")
OUTPUT_DIR = Path("outputs/dragonfly_gr").resolve()

PIXEL_SCALE = 2.5  # arcsec/pixel
PSF_FWHM = 6.0     # arcsec
ZEROPOINT_G = 27.5
ZEROPOINT_R = 27.5

## Configuration

RHT remains the default morphology backend. Set `morphology.backend` to `starlet` to use the optional multiscale reconstruction.

In [ ]:
config = ModelingConfig.from_dict({
    "run": {
        "name": "dragonfly_gr",
        "output_dir": str(OUTPUT_DIR),
        "random_seed": 12345,
    },
    "bands": {
        "g": {
            "image": str(G_IMAGE),
            "wavelength": 4770,
            "pixel_scale": PIXEL_SCALE,
            "psf_fwhm": PSF_FWHM,
            "zeropoint": ZEROPOINT_G,
        },
        "r": {
            "image": str(R_IMAGE),
            "wavelength": 6231,
            "pixel_scale": PIXEL_SCALE,
            "psf_fwhm": PSF_FWHM,
            "zeropoint": ZEROPOINT_R,
        },
    },
    "planck_radiance": {"path": str(PLANCK_MAP)},
    "alignment": {"reference_band": "r", "psf_match": True},
    "morphology": {
        "backend": "rht",
        "working_pixel_scale": 10,
        "rht": {
            "radius": 3,
            "response": "peak",
            "angle_bins": "auto",
            "median_filter_size": 3,
        },
    },
    "color": {
        "reference_band": "r",
        "bands": ["g", "r"],
        "relation": "linear",
        "regression": "bisector",
        "combine": "median",
        "bootstrap_samples": 100,
        "fit_sigma_range": [-15, 20],
    },
})
config.validate_files()

## Run the model

In [ ]:
modeler = MultiBandModeler(config)
result = modeler.run()

## Cirrus color

The reported uncertainty is estimated by bootstrapping the Planck correlations.

In [ ]:
color = result.colors["g-r"]
print(f"g-r = {color.value:.3f} +/- {color.error:.3f} mag")

## Goodness of the Planck and inter-band fits

In [ ]:
plot_planck_fits(result);

In [ ]:
plot_interband_fits(result);

## RHT diagnostics

These panels show the working luminance image, directional response, residual structure, compact-source mask, and filtered output when available.

In [ ]:
plot_morphology(result);

## Main model images

In [ ]:
plot_model_images(result);

luminance_image = result.luminance
g_model = result.models["g"]
r_model = result.models["r"]

## Save outputs

In [ ]:
result.write(OUTPUT_DIR, overwrite=True)
print(f"Saved outputs to {OUTPUT_DIR.resolve()}")